# NGUYỄN THÀNH TRÍ - MSSV: 6451071079
## BÀI 3: PHÂN LỚP ẢNH CHỮ SỐ VIẾT TAY MNIST BẰNG NEURAL NETWORK KẾT HỢP 10-FOLD CROSS VALIDATION
### (K-FOLD CROSS VALIDATION VỚI K = 10)

In [1]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import keras
from keras import layers, models

print("Đã nạp đầy đủ các thư viện cần thiết cho Bài 3.")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


I0000 00:00:1794280021.947837   59588 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Đã nạp đầy đủ các thư viện cần thiết cho Bài 3.


I0000 00:00:1794280023.204125   59588 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Câu a: Đọc bộ dữ liệu train, test của MNIST từ Keras
Hướng dẫn: Sử dụng keras.datasets.mnist.load_data() để đọc dữ liệu ảnh và nhãn.

In [2]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()
print("Đã tải thành công bộ dữ liệu MNIST từ Keras.")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Đã tải thành công bộ dữ liệu MNIST từ Keras.


### Câu b: In ra kích thước của tập train và test
Hướng dẫn: Sử dụng thuộc tính .shape của các mảng dữ liệu ảnh và nhãn.

In [3]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

print(f"Kích thước tập ảnh train: {train_images.shape}")
print(f"Kích thước tập nhãn train: {train_labels.shape}")
print(f"Kích thước tập ảnh test:  {test_images.shape}")
print(f"Kích thước tập nhãn test:  {test_labels.shape}")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Kích thước tập ảnh train: (60000, 28, 28)
Kích thước tập nhãn train: (60000,)
Kích thước tập ảnh test:  (10000, 28, 28)
Kích thước tập nhãn test:  (10000,)


### Câu c: Chuẩn hóa các giá trị pixel của ảnh về đoạn [0, 1]
Hướng dẫn: Chia các giá trị pixel cho 255.0 để đưa cường độ sáng về đoạn số thực [0, 1].

In [4]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

print(f"Giá trị pixel nhỏ nhất sau chuẩn hóa: {train_images.min():.1f}")
print(f"Giá trị pixel lớn nhất sau chuẩn hóa: {train_images.max():.1f}")
print(f"Kiểu dữ liệu của mảng ảnh: {train_images.dtype}")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


Giá trị pixel nhỏ nhất sau chuẩn hóa: 0.0
Giá trị pixel lớn nhất sau chuẩn hóa: 1.0
Kiểu dữ liệu của mảng ảnh: float32


### Câu d: Định nghĩa hàm create_model khởi tạo và biên dịch mạng Neural Network
Kiến trúc mô hình:
- Flatten layer: Làm phẳng ma trận 2D (28x28) thành vector 784 chiều.
- Dense layer: 128 nơ-ron, hàm kích hoạt ReLU.
- Dropout layer: Tỷ lệ 0.2.
- Dense layer (Output): 10 nơ-ron, hàm kích hoạt Softmax.
- Biên dịch với: optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'].

In [5]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def create_model():
    model = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Khởi tạo mô hình thử nghiệm và in cấu trúc
sample_model = create_model()
sample_model.summary()

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)